# 🏥 New Relic Health Assessment

This notebook provides an interactive interface to:
1. **Configure** your target app and time window
2. **Download** all metrics from New Relic (performance, errors, infrastructure, logs, etc.)
3. **Calculate** a weighted health score across 5 categories
4. **Display** the full health report inline

---
> **Pre-requisite:** ensure `.newrelic_config.dev.json` or `.newrelic_config.prod.json` exists in the project root with your API key, account ID, and app ID.  
> See `.newrelic_config.example.json` for the expected format.

---
## ⚙️ Step 1 — Configure
Set your **profile** and **time window** here, then run the cell.

In [ ]:
# ============================================================
#  EDIT THESE VALUES BEFORE RUNNING
# ============================================================

PROFILE = "dev"   # "dev" or "prod"  → loads .newrelic_config.{PROFILE}.json
DAYS    = 1       # 1 | 3 | 7 | 14 | 30  (days of data to pull)

# ============================================================
import sys, os
# Make sure project root is on the path
PROJECT_ROOT = os.path.abspath(os.path.dirname("__file__") if "__file__" in dir() else ".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from modules.config_loader import load_config

config = load_config(profile=PROFILE, days=DAYS)

APP_ID   = config["app_ids"][0]
APP_NAME = config.get("app_name", f"app-{APP_ID}")

print(f"✓ Profile:     {PROFILE}")
print(f"✓ Account ID:  {config['account_id']}")
print(f"✓ App:         {APP_NAME}  (id: {APP_ID})")
print(f"✓ Time window: last {DAYS} day(s)")

---
## 📡 Step 2 — Download All Metrics from New Relic
Pulls 14 data categories in sequence.  Expect **1–3 minutes** depending on API latency.

In [ ]:
import json, os
from datetime import datetime, timedelta, timezone
try:
    from zoneinfo import ZoneInfo
    _EST = ZoneInfo("America/New_York")
except Exception:
    _EST = timezone(timedelta(hours=-5))
from modules.api_client import ApiClient

DATA_DIR = "data"

client = ApiClient(
    api_key=config["api_key"],
    account_id=config["account_id"],
    timeout=config.get("timeout", 30),
    max_retries=config.get("max_retries", 2),
)

def fetch(label, fn, *args, **kwargs):
    print(f"  ⏳ {label} ...", end=" ", flush=True)
    result = fn(*args, **kwargs)
    print("✓")
    return result

print(f"Fetching data for '{APP_NAME}'  (last {DAYS} day(s))\n")
collection_start = datetime.now(_EST)

performance_data      = fetch("Performance metrics (P50/P95/P99, Apdex, throughput)",
                               client.fetch_performance_metrics, APP_ID, DAYS)
error_data            = fetch("Error metrics",
                               client.fetch_error_metrics, APP_ID, DAYS)
infrastructure_data   = fetch("Infrastructure (CPU, memory)",
                               client.fetch_infrastructure_metrics, APP_ID, DAYS, app_name=APP_NAME)
database_data         = fetch("Database summary",
                               client.fetch_database_metrics, APP_ID, DAYS)
transaction_data      = fetch("API / transaction metrics",
                               client.fetch_transaction_metrics, APP_ID, DAYS)
error_details_data    = fetch("Error details & stack traces",
                               client.fetch_error_details, APP_ID, DAYS)
slow_tx_data          = fetch("Slow transactions (top 20)",
                               client.fetch_slow_transactions, APP_ID, DAYS)
db_details_data       = fetch("Database query details",
                               client.fetch_database_details, APP_ID, DAYS)
slow_db_tx_data       = fetch("Top-20 slowest database transactions",
                               client.fetch_slow_db_transactions, APP_ID, DAYS)
external_svc_data     = fetch("External service breakdown",
                               client.fetch_external_services, APP_ID, DAYS)
application_logs_data = fetch("Application logs (errors / warnings / exceptions)",
                               client.fetch_application_logs, APP_ID, DAYS, app_name=APP_NAME)
log_volume_data       = fetch("Log volume by level",
                               client.fetch_log_volume, APP_ID, DAYS, app_name=APP_NAME)
alerts_data           = fetch("Alert / incident history",
                               client.fetch_alerts, APP_ID, DAYS, app_name=APP_NAME)
hourly_trends_data    = fetch("Hourly performance trends",
                               client.fetch_hourly_trends, APP_ID, DAYS)
baselines_data        = fetch("7-day baselines",
                               client.fetch_baselines, APP_ID)
deployments_data      = fetch("Deployment markers",
                               client.fetch_deployments, APP_ID, DAYS, app_name=APP_NAME)
collection_end = datetime.now(_EST)

# ── Assemble full payload ──────────────────────────────────────────────────
all_data = {
    "app_id":             APP_ID,
    "app_name":           APP_NAME,
    "days":               DAYS,
    "collected_at":       datetime.now(_EST).isoformat(),
    "performance":        performance_data,
    "errors":             error_data,
    "infrastructure":     infrastructure_data,
    "database":           database_data,
    "transactions":       transaction_data,
    "error_details":      error_details_data,
    "slow_transactions":  slow_tx_data,
    "database_details":   db_details_data,
    "slow_db_transactions": slow_db_tx_data,
    "external_services":  external_svc_data,
    "application_logs":   application_logs_data,
    "log_volume":         log_volume_data,
    "alerts":             alerts_data,
    "hourly_trends":      hourly_trends_data,
    "baselines":          baselines_data,
    "deployments":        deployments_data,
}

# ── Save raw JSON ──────────────────────────────────────────────────────────
os.makedirs(DATA_DIR, exist_ok=True)
ts = datetime.now(_EST).strftime("%Y-%m-%d-%H%M%S")
cache_file = os.path.join(DATA_DIR, f"{APP_NAME}-{DAYS}d-{ts}.json")
with open(cache_file, "w", encoding="utf-8") as f:
    json.dump(all_data, f, indent=2)


print(f"\n✅ All data collected and saved → {cache_file}")
print(f"\n✅ All data collected and saved → {cache_file}")

---
## 🧮 Step 3 — Calculate Health Score

In [ ]:
from modules.health_calculator import HealthCalculator

calculator = HealthCalculator()

# Flatten nested metric dicts into a single dict for the calculator
flat_metrics = {}
flat_metrics.update(performance_data.get("performance", {}))
flat_metrics.update(error_data.get("errors", {}))
flat_metrics.update(infrastructure_data.get("infrastructure", {}))
flat_metrics.update(database_data.get("database", {}))
flat_metrics.update(transaction_data.get("transactions", {}))

health_data = calculator.calculate_health_score(flat_metrics)

score   = health_data["overall_score"]
status  = health_data["status"]
scores  = health_data.get("category_scores", {})
findings = health_data.get("findings", [])

status_emoji = {"Healthy": "🟢", "Warning": "🟡", "Critical": "🔴"}.get(status, "⚪")

print(f"{status_emoji}  Overall Score: {score}/100  ({status})")
print()
print("Category Breakdown:")
category_labels = {
    "performance":     "Performance   (25%)",
    "errors":          "Errors        (25%)",
    "infrastructure":  "Infrastructure(20%)",
    "database":        "Database      (15%)",
    "api":             "API           (15%)",
}
for key, label in category_labels.items():
    s = scores.get(key, "N/A")
    bar = "█" * (s // 10) + "░" * (10 - s // 10) if isinstance(s, int) else ""
    print(f"  {label}  {bar}  {s}")

critical = [f for f in findings if f["severity"] == "Critical"]
warnings = [f for f in findings if f["severity"] == "Warning"]
print()
print(f"Findings: {len(critical)} Critical  |  {len(warnings)} Warning")
for f in critical:
    print(f"  🔴 [{f['category']}] {f.get('issue', '')}: {f.get('description', '')}")
for f in warnings:
    print(f"  🟡 [{f['category']}] {f.get('issue', '')}: {f.get('description', '')}")

---
## 📄 Step 4 — Generate & Display Health Report

In [ ]:
from IPython.display import Markdown, display
from modules.report_generator import ReportGenerator

generator = ReportGenerator()

report_config = {
    "app_name":   APP_NAME,
    "account_id": config["account_id"],
    "app_id":     APP_ID,
    "days":       DAYS,
}

report = generator.generate_report(
    health_data, APP_ID, report_config, metrics=all_data,
    collection_start=collection_start, collection_end=collection_end
)

# Save to disk
filepath = generator.save_report(report, app_id=APP_NAME)
print(f"✅ Report saved → {filepath}\n")

# Render inline
display(Markdown(report))

---
## 🔍 Step 5 (Optional) — Inspect Raw Data

In [ ]:
# ── Application Logs (errors / warnings) ──────────────────────────────────
import json

logs = all_data.get("application_logs", {}).get("logs", [])
print(f"Application log entries captured: {len(logs)}")
for entry in logs[:20]:   # show first 20
    ts_  = entry.get("timestamp", "")
    lvl  = entry.get("level", "").upper()
    msg  = entry.get("message", "")[:120]
    print(f"  [{lvl:7s}] {ts_}  {msg}")

In [ ]:
# ── Slow Transactions ──────────────────────────────────────────────────────
slow = all_data.get("slow_transactions", {}).get("slow_transactions", [])
print(f"Slow transactions captured: {len(slow)}")
print(f"{'Transaction':<55} {'Avg ms':>8} {'P95 ms':>8} {'Calls':>7}")
print("-" * 82)
for tx in slow[:20]:
    name = tx.get("name", "")[-55:]
    avg   = tx.get("avg_duration_ms", 0) or 0
    p95   = tx.get("p95_ms", 0) or 0
    calls = tx.get("call_count", 0) or 0
    print(f"{name:<55} {avg:>8.0f} {p95:>8.0f} {calls:>7}")

In [ ]:
# ── Error Details ──────────────────────────────────────────────────────────
errs = all_data.get("error_details", {}).get("errors", [])
print(f"Error class entries captured: {len(errs)}")
print(f"{'Error Class':<50} {'Count':>6} {'Sample Message':<60}")
print("-" * 120)
for e in errs[:20]:
    cls  = e.get("error_class", "")[:50]
    cnt  = e.get("count", 0)
    msg  = e.get("message", "")[:60]
    print(f"{cls:<50} {cnt:>6} {msg:<60}")

In [ ]:
# ── Load data from a previously saved JSON file (re-run without API calls) ─
import glob, json, os

files = sorted(glob.glob("data/*.json"), key=os.path.getmtime, reverse=True)
if files:
    print("Saved data files (newest first):")
    for i, f in enumerate(files):
        size_kb = os.path.getsize(f) / 1024
        print(f"  [{i}] {os.path.basename(f)}  ({size_kb:.1f} KB)")
    print()
    # Uncomment and set index to reload a specific file:
    # LOAD_INDEX = 0
    # with open(files[LOAD_INDEX], encoding="utf-8") as fh:
    #     all_data = json.load(fh)
    # print(f"Loaded: {files[LOAD_INDEX]}")
else:
    print("No saved data files found in data/ folder.")